In [ ]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm import tqdm

OUTPUT_DIR = Path("../data/audio_processed/")

# RAVDESS encodes emotion in the 3rd part of the filename
# e.g. 03-01-05-01-01-01-01.wav → position 2 = '05' = angry
RAVDESS_EMOTION_MAP = {
    '01':'neutral', '02':'calm',    '03':'happy',
    '04':'sad',     '05':'angry',   '06':'fearful',
    '07':'disgust', '08':'surprised'
}

RISK_MAP = {
    'neutral':'Low',   'calm':'Low',     'happy':'Low',
    'fearful':'Moderate', 'angry':'Moderate', 'disgust':'Moderate',
    'sad':'High',      'surprised':'Low'
}

def extract_features(file_path: Path) -> np.ndarray | None:
    """
    Extracts a fixed-length 162-number vector from one audio clip.

    Features extracted:
    - 40 MFCCs  (mean + std = 80 numbers)  → captures tone/timbre
    - 12 Chroma (mean + std = 24 numbers)  → captures pitch/harmony
    - 7  Spectral contrast (mean + std = 14 numbers) → captures richness
    - 1  Pitch/F0 mean                      → fundamental frequency
    - 1  Pitch std                          → pitch variability
    - 1  RMS energy mean                    → loudness
    - 1  RMS std                            → loudness variability
    Total: 162 features
    """
    try:
        y, sr = librosa.load(file_path, sr=22050, mono=True)
        if len(y) < sr * 0.5:      # skip clips shorter than 0.5s
            return None

        # MFCCs — most important feature for speech emotion
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        mfcc_mean = mfcc.mean(axis=1)   # 40 numbers
        mfcc_std  = mfcc.std(axis=1)    # 40 numbers

        # Chroma — pitch class energy
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = chroma.mean(axis=1)   # 12 numbers
        chroma_std  = chroma.std(axis=1)    # 12 numbers

        # Spectral contrast — difference between peaks and valleys
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = contrast.mean(axis=1)   # 7 numbers
        contrast_std  = contrast.std(axis=1)    # 7 numbers

        # Pitch (fundamental frequency)
        f0, _, _ = librosa.pyin(y, fmin=50, fmax=400,
                                 sr=sr, fill_na=0.0)
        pitch_mean = np.mean(f0)    # 1 number
        pitch_std  = np.std(f0)     # 1 number

        # RMS energy (loudness)
        rms = librosa.feature.rms(y=y)
        rms_mean = rms.mean()       # 1 number
        rms_std  = rms.std()        # 1 number

        # Stack all into one flat vector
        return np.concatenate([
            mfcc_mean, mfcc_std,
            chroma_mean, chroma_std,
            contrast_mean, contrast_std,
            [pitch_mean, pitch_std, rms_mean, rms_std]
        ])  # → 162 total features

    except Exception as e:
        print(f"Failed: {file_path.name} — {e}")
        return None


# ── Run extraction on all files ──────────────────────────
wav_files = list(OUTPUT_DIR.rglob("*.wav"))
print(f"Extracting features from {len(wav_files)} files...")

rows = []
for f in tqdm(wav_files, desc="Audio feature extraction"):
    parts = f.stem.split('-')
    if len(parts) < 3:
        continue

    emotion_code = parts[2]
    emotion = RAVDESS_EMOTION_MAP.get(emotion_code)
    risk    = RISK_MAP.get(emotion)
    if not risk:
        continue

    feat = extract_features(f)
    if feat is not None:
        rows.append({'file': f.name, 'emotion': emotion,
                     'risk_level': risk, **{f'f{i}': v
                     for i, v in enumerate(feat)}})

df_audio = pd.DataFrame(rows)

# ── Save ─────────────────────────────────────────────────
Path('../data/features').mkdir(parents=True, exist_ok=True)

feature_cols = [c for c in df_audio.columns if c.startswith('f')]
X_audio = df_audio[feature_cols].values
y_audio = df_audio['risk_level'].values

np.save('../data/features/X_audio.npy', X_audio)
np.save('../data/features/y_audio.npy', y_audio)
df_audio.to_csv('../data/features/audio_features.csv', index=False)

print(f"\n✅ Audio features saved.")
print(f"   Feature matrix shape : {X_audio.shape}")
# Expected: (1440, 162)
print(f"   Label counts : {pd.Series(y_audio).value_counts().to_dict()}")

Extracting features from 2452 files...


Audio feature extraction: 100%|██████████| 2452/2452 [09:51<00:00,  4.15it/s]



✅ Audio features saved.
   Feature matrix shape : (2452, 123)
   Label counts : {'Low': 1132, 'Moderate': 944, 'High': 376}
